In [59]:
# --- 1: A INSPEÇÃO DE TODOS OS ARQUIVOS LIMPOS ---

# 1. Importar a caixa de ferramentas
import pandas as pd

# 2. Definir a pasta onde estão meus arquivos limpos
pasta_finalizados = r'C:\Users\fabio\TCC\FINALIZADOS'

# 3. Listar o nome EXATO de todos os 11 arquivos
nomes_arquivos = [
    '1.CFEM_FINAL.csv',
    '2_arrecadacao_FINAL.csv',
    '3_populacao_FINAL.csv',
    '4_PIB_FINAL.csv',
    '5_VAB_IND_FINAL.csv',
    '6_VAB_SERV_FINAL.csv',
    '7_emprego_total_FINAL.csv',
    '8_Emprego_Min_FINAL.csv',
    '9_emprego_serv_FINAL.csv',
    '10_Mort_FINAL.csv',
    '11_peso_FINAL.csv'
]

# 4. Um "dicionário" para guardar todas as tabelas carregadas
dataframes_carregados = {}

print("--- INICIANDO INSPEÇÃO GERAL DAS COLUNAS ---")

# 5. O Loop de Inspeção
for nome in nomes_arquivos:
    
    # 6. Montar o caminho completo
    caminho = f'{pasta_finalizados}\\{nome}'
    
    try:
        # 7. Carregar o arquivo CSV
        df = pd.read_csv(caminho)
        
        # 8. Guardar o dataframe no "fichário"
        dataframes_carregados[nome] = df
        
        # 9. IMPRIMIR AS COLUNAS
        print(f"\nArquivo: {nome}")
        print(f"Colunas: {df.columns.tolist()}")
        
    except FileNotFoundError:
        print(f"\nERRO: Arquivo {nome} não encontrado em {pasta_finalizados}")
    except Exception as e:
        print(f"\nERRO ao ler {nome}: {e}")

print("\n--- Inspeção Concluída ---")

--- INICIANDO INSPEÇÃO GERAL DAS COLUNAS ---

Arquivo: 1.CFEM_FINAL.csv
Colunas: ['Ano', 'CodigoMunicipio', 'Município', 'CFEM']

Arquivo: 2_arrecadacao_FINAL.csv
Colunas: ['MUNICIPIOS', 'UF', 'Ano', 'Arrecadacao']

Arquivo: 3_populacao_FINAL.csv
Colunas: ['Sigla', 'Código', 'Município', 'Ano', 'Populacao']

Arquivo: 4_PIB_FINAL.csv
Colunas: ['Sigla', 'Código', 'Município', 'Ano', 'PIB']

Arquivo: 5_VAB_IND_FINAL.csv
Colunas: ['Sigla', 'Código', 'Município', 'Ano', 'VAB Ind.']

Arquivo: 6_VAB_SERV_FINAL.csv
Colunas: ['Sigla', 'Código', 'Município', 'Ano', 'VAB Serv.']

Arquivo: 7_emprego_total_FINAL.csv
Colunas: ['Estado', 'Cidade', 'Ano', 'Vinculos_totais']

Arquivo: 8_Emprego_Min_FINAL.csv
Colunas: ['Sigla_UF', 'Nome_Municipio', 'Ano', 'Empregos_mineracao']

Arquivo: 9_emprego_serv_FINAL.csv
Colunas: ['Sigla_UF', 'Nome_Municipio', 'Ano', 'Empregos_servicos']

Arquivo: 10_Mort_FINAL.csv
Colunas: ['Sigla_UF', 'Nome_Municipio', 'Ano', 'Mort_infant']

Arquivo: 11_peso_FINAL.csv
Colunas: 

In [60]:
# --- 2: INSTALANDO O 'unidecode' ---

# O '!' diz ao Jupyter para rodar este comando no terminal
!pip install unidecode

print("Ferramenta 'unidecode' instalada com sucesso!")

Ferramenta 'unidecode' instalada com sucesso!


In [61]:
# --- 3: A "TABELA MESTRA" ---

# 1. Importar as ferramentas
import pandas as pd
from unidecode import unidecode

# 2. Carregar o arquivo de População
pasta_finalizados = r'C:\Users\fabio\TCC\FINALIZADOS'
caminho_populacao = f'{pasta_finalizados}\\3_populacao_FINAL.csv'
df_pop = pd.read_csv(caminho_populacao)

# 3. Criar a tabela mestra (selecionar/remover duplicatas)
colunas_chave = ['Código', 'Município', 'Sigla']
df_mestra_codigos = df_pop[colunas_chave].drop_duplicates()

# 4. Renomear as colunas para o padrão
df_mestra_codigos = df_mestra_codigos.rename(columns={
    'Código': 'Cod_IBGE',
    'Município': 'Nome_Mestre',
    'Sigla': 'UF_Mestre'
})

# 5. Criar as "Chaves de Junção Limpas"
df_mestra_codigos['key_nome'] = df_mestra_codigos['Nome_Mestre'].apply(lambda x: unidecode(str(x)).upper().strip())
df_mestra_codigos['key_uf'] = df_mestra_codigos['UF_Mestre'].apply(lambda x: unidecode(str(x)).upper().strip())
df_mestra_codigos['UF_Code_Mestre'] = df_mestra_codigos['Cod_IBGE'].astype(str).str[:2]

# 6. [A CORREÇÃO DAS DUPLICATAS] Remover ambiguidades de nome/UF
df_mestra_codigos = df_mestra_codigos.drop_duplicates(subset=['key_nome', 'key_uf'], keep='first')

print("--- Tabela Mestra de Códigos (À PROVA DE FALHAS) CRIADA ---")
print("Ela agora é 100% única nas chaves de nome e UF.")

--- Tabela Mestra de Códigos (À PROVA DE FALHAS) CRIADA ---
Ela agora é 100% única nas chaves de nome e UF.


In [62]:
# --- 4: AGREGANDO CFEM E EMPREGO ---

# (Recarregar os 11 arquivos na memória)
dataframes_carregados = {}
nomes_arquivos = [
    '1.CFEM_FINAL.csv','2_arrecadacao_FINAL.csv','3_populacao_FINAL.csv',
    '4_PIB_FINAL.csv','5_VAB_IND_FINAL.csv','6_VAB_SERV_FINAL.csv',
    '7_emprego_total_FINAL.csv','8_Emprego_Min_FINAL.csv',
    '9_emprego_serv_FINAL.csv','10_Mort_FINAL.csv','11_peso_FINAL.csv'
]
pasta_finalizados = r'C:\Users\fabio\TCC\FINALIZADOS'
for nome in nomes_arquivos:
    caminho = f'{pasta_finalizados}\\{nome}'
    dataframes_carregados[nome] = pd.read_csv(caminho)
print("--- Todos os 11 arquivos recarregados na memória ---")

# (Recarregar a Tabela Mestra de Códigos)
df_pop_mestra = dataframes_carregados['3_populacao_FINAL.csv']
colunas_chave = ['Código', 'Município', 'Sigla']
df_mestra_codigos = df_pop_mestra[colunas_chave].drop_duplicates()
df_mestra_codigos = df_mestra_codigos.rename(columns={'Código': 'Cod_IBGE', 'Município': 'Nome_Mestre', 'Sigla': 'UF_Mestre'})
df_mestra_codigos['Cod_IBGE'] = df_mestra_codigos['Cod_IBGE'].astype(int).astype(str).str[:6] 
df_mestra_codigos['key_nome'] = df_mestra_codigos['Nome_Mestre'].apply(lambda x: unidecode(str(x)).upper().strip())
df_mestra_codigos['key_uf'] = df_mestra_codigos['UF_Mestre'].apply(lambda x: unidecode(str(x)).upper().strip())
print("--- Tabela Mestra de Códigos (com Cod_IBGE de 6 DÍGITOS) RECARREGADA ---")

# --- Início da Padronização ---
lista_dfs_padronizados = []
print("--- Padronizando arquivos (COM LÓGICA DE 6 DÍGITOS) ---")

# (Função de ajuda para forçar chaves para string e CORTAR o código)
def forcar_chaves_para_string(df, col_cod, col_ano):
    df = df.dropna(subset=[col_cod, col_ano])
    df['Cod_IBGE'] = df[col_cod].astype(float).astype(int).astype(str).str[:6]
    df['Ano'] = df[col_ano].astype(float).astype(int).astype(str)
    return df

# --- [CORREÇÃO] Arquivo 1 (CFEM) ---
print("Processando CFEM (COM AGREGAÇÃO)...")
df_cfem = dataframes_carregados['1.CFEM_FINAL.csv'].dropna(subset=['CodigoMunicipio', 'Ano'])
df_cfem['Cod_IBGE'] = df_cfem['CodigoMunicipio'].astype(float).astype(int).astype(str).str[:6]
df_cfem['Ano'] = df_cfem['Ano'].astype(str)
# 1. Agrupar e Somar o CFEM por (Cod_IBGE, Ano)
df_cfem_agregado = df_cfem.groupby(['Cod_IBGE', 'Ano'])['CFEM'].sum().reset_index()
lista_dfs_padronizados.append(df_cfem_agregado)

# (População)
df_pop = dataframes_carregados['3_populacao_FINAL.csv']
df_pop_padrao = forcar_chaves_para_string(df_pop, 'Código', 'Ano')
df_pop_padrao = df_pop_padrao.rename(columns={'Município':'Nome_Municipio_Mestre', 'Sigla':'Sigla_Mestre'})
lista_dfs_padronizados.append(df_pop_padrao[['Cod_IBGE', 'Nome_Municipio_Mestre', 'Sigla_Mestre', 'Ano', 'Populacao']])

# (PIB, VAB Ind, VAB Serv)
df_pib = dataframes_carregados['4_PIB_FINAL.csv']
df_pib_padrao = forcar_chaves_para_string(df_pib, 'Código', 'Ano')
lista_dfs_padronizados.append(df_pib_padrao[['Cod_IBGE', 'Ano', 'PIB']])
df_vab_ind = dataframes_carregados['5_VAB_IND_FINAL.csv']
df_vab_ind_padrao = forcar_chaves_para_string(df_vab_ind, 'Código', 'Ano')
lista_dfs_padronizados.append(df_vab_ind_padrao[['Cod_IBGE', 'Ano', 'VAB Ind.']])
df_vab_serv = dataframes_carregados['6_VAB_SERV_FINAL.csv']
df_vab_serv_padrao = forcar_chaves_para_string(df_vab_serv, 'Código', 'Ano')
lista_dfs_padronizados.append(df_vab_serv_padrao[['Cod_IBGE', 'Ano', 'VAB Serv.']])
print("Arquivos fáceis padronizados.")

# --- Padronizando arquivos 'DIFÍCEIS' ---
print("\n--- Padronizando arquivos 'DIFÍCEIS' (COM UNIDECODE E 6 DÍGITOS) ---")

# (Função de ajuda de limpeza pesada)
def consertar_arquivo_pesado(df, col_nome, col_uf, col_ano, col_valor):
    df = df.rename(columns={col_nome: 'key_nome_sujo', col_uf: 'key_uf_sujo', col_ano: 'Ano', col_valor: 'Valor'})
    df['key_nome'] = df['key_nome_sujo'].apply(lambda x: unidecode(str(x)).upper().strip())
    df['key_uf'] = df['key_uf_sujo'].apply(lambda x: unidecode(str(x)).upper().strip())
    df['Ano'] = df['Ano'].astype(int).astype(str)
    df_com_codigo = pd.merge(df, df_mestra_codigos, on=['key_nome', 'key_uf'])
    # [CORREÇÃO] AGREGAR (SOMAR) OS VALORES
    df_agregado = df_com_codigo.groupby(['Cod_IBGE', 'Ano'])['Valor'].sum().reset_index()
    # Renomear a coluna 'Valor' de volta para o nome original
    df_agregado = df_agregado.rename(columns={'Valor': col_valor})
    return df_agregado

# (Arrecadação)
df_arr = dataframes_carregados['2_arrecadacao_FINAL.csv']
df_arr_consertado = consertar_arquivo_pesado(df_arr, 'MUNICIPIOS', 'UF', 'Ano', 'Arrecadacao')
lista_dfs_padronizados.append(df_arr_consertado)

# (Emprego Total)
df_emp_tot = dataframes_carregados['7_emprego_total_FINAL.csv']
df_emp_tot_consertado = consertar_arquivo_pesado(df_emp_tot, 'Cidade', 'Estado', 'Ano', 'Vinculos_totais')
lista_dfs_padronizados.append(df_emp_tot_consertado)

# (Emprego Min)
df_emp_min = dataframes_carregados['8_Emprego_Min_FINAL.csv']
df_emp_min_consertado = consertar_arquivo_pesado(df_emp_min, 'Nome_Municipio', 'Sigla_UF', 'Ano', 'Empregos_mineracao')
lista_dfs_padronizados.append(df_emp_min_consertado)

# (Emprego Serv)
df_emp_serv = dataframes_carregados['9_emprego_serv_FINAL.csv']
df_emp_serv_consertado = consertar_arquivo_pesado(df_emp_serv, 'Nome_Municipio', 'Sigla_UF', 'Ano', 'Empregos_servicos')
lista_dfs_padronizados.append(df_emp_serv_consertado)

# (Mortalidade)
print("Processando Mortalidade Infantil (COM LÓGICA DE 6 DÍGITOS)...")
df_mort = dataframes_carregados['10_Mort_FINAL.csv']
df_mort_padrao = forcar_chaves_para_string(df_mort, col_cod='Sigla_UF', col_ano='Ano')
df_mort_final = df_mort_padrao[['Cod_IBGE', 'Ano', 'Mort_infant']]
lista_dfs_padronizados.append(df_mort_final)

print("\n--- TODOS OS ARQUIVOS FORAM PADRONIZADOS E AGREGADOS ---")

--- Todos os 11 arquivos recarregados na memória ---
--- Tabela Mestra de Códigos (com Cod_IBGE de 6 DÍGITOS) RECARREGADA ---
--- Padronizando arquivos (COM LÓGICA DE 6 DÍGITOS) ---
Processando CFEM (COM AGREGAÇÃO)...


C:\Users\fabio\AppData\Local\Temp\ipykernel_11292\125822856.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cfem['Cod_IBGE'] = df_cfem['CodigoMunicipio'].astype(float).astype(int).astype(str).str[:6]
C:\Users\fabio\AppData\Local\Temp\ipykernel_11292\125822856.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cfem['Ano'] = df_cfem['Ano'].astype(str)


Arquivos fáceis padronizados.

--- Padronizando arquivos 'DIFÍCEIS' (COM UNIDECODE E 6 DÍGITOS) ---
Processando Mortalidade Infantil (COM LÓGICA DE 6 DÍGITOS)...

--- TODOS OS ARQUIVOS FORAM PADRONIZADOS E AGREGADOS ---


C:\Users\fabio\AppData\Local\Temp\ipykernel_11292\125822856.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Cod_IBGE'] = df[col_cod].astype(float).astype(int).astype(str).str[:6]
C:\Users\fabio\AppData\Local\Temp\ipykernel_11292\125822856.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Ano'] = df[col_ano].astype(float).astype(int).astype(str)


In [63]:
# --- 5: A JUNÇÃO MESTRA ---

# 1. Pegar a base (CFEM AGREGADO)
df_base_final = lista_dfs_padronizados[0].copy()

# 2. O Loop de Junção (para os 9 arquivos restantes)
for df_para_adicionar in lista_dfs_padronizados[1:]:
    
    # 3. Juntar (merge) com a base
    df_base_final = pd.merge(
        df_base_final, 
        df_para_adicionar, 
        on=['Cod_IBGE', 'Ano'], 
        how='outer'
    )
    print(f"Juntou... Nova base tem {df_base_final.shape[1]} colunas.")

print("\n--- JUNÇÃO MESTRA (DEFINITIVA) CONCLUÍDA ---")

Juntou... Nova base tem 6 colunas.
Juntou... Nova base tem 7 colunas.
Juntou... Nova base tem 8 colunas.
Juntou... Nova base tem 9 colunas.
Juntou... Nova base tem 10 colunas.
Juntou... Nova base tem 11 colunas.
Juntou... Nova base tem 12 colunas.
Juntou... Nova base tem 13 colunas.
Juntou... Nova base tem 14 colunas.

--- JUNÇÃO MESTRA (DEFINITIVA) CONCLUÍDA ---


In [64]:
# --- 6: OS TESTES ---

# 1. Ordenar a Tabela Mestra
print("Ordenando a tabela final...")
df_base_final_ordenada = df_base_final.sort_values(by=['Cod_IBGE', 'Ano'])
print("Tabela ordenada.")

# --- TESTE 1: O "SPOT-CHECK" EM UBERLÂNDIA ---
codigo_uberlandia = '317020' # Código de 6 DÍGITOS
df_uberlandia = df_base_final_ordenada[
    df_base_final_ordenada['Cod_IBGE'] == codigo_uberlandia
]

print(f"\n--- MOSTRANDO DADOS PARA UBERLÂNDIA (Cód: {codigo_uberlandia}) ---")
print(f"Encontradas {len(df_uberlandia)} linhas de dados.")
print(df_uberlandia) 

# --- TESTE 2: O TESTE DE INTEGRIDADE (DUPLICATAS) ---
chaves_do_painel = ['Cod_IBGE', 'Ano']
numero_de_duplicatas = df_base_final_ordenada.duplicated(subset=chaves_do_painel).sum()

print("\n--- VERIFICANDO INTEGRIDIDE DOS DADOS ---")
print(f"Número de linhas duplicadas (mesmo Cod_IBGE e Ano): {numero_de_duplicatas}")

if numero_de_duplicatas == 0:
    print("SUCESSO! Seus dados são únicos. A base de painel é válida.")
else:
    print(f"ATENÇÃO! Você tem {numero_de_duplicatas} linhas duplicadas.")

Ordenando a tabela final...
Tabela ordenada.

--- MOSTRANDO DADOS PARA UBERLÂNDIA (Cód: 317020) ---
Encontradas 35 linhas de dados.
       Cod_IBGE   Ano      CFEM Nome_Municipio_Mestre Sigla_Mestre  Populacao  \
105937   317020  1985       NaN                   NaN          NaN        NaN   
105938   317020  1992       NaN            Uberlândia           MG  377933.00   
105939   317020  1993       NaN            Uberlândia           MG  388483.00   
105940   317020  1994       NaN            Uberlândia           MG  398216.00   
105941   317020  1995       NaN            Uberlândia           MG  407707.00   
105942   317020  1996       NaN                   NaN          NaN        NaN   
105943   317020  1997       NaN            Uberlândia           MG  456920.00   
105944   317020  1998       NaN            Uberlândia           MG  472083.00   
105945   317020  1999       NaN            Uberlândia           MG  487222.00   
105946   317020  2000       NaN            Uberlândia     

In [65]:
# --- 7: O POLIMENTO FINAL (PREENCHENDO NOMES E UFs) ---

# 1. Pegar a base final
df_final = df_base_final_ordenada.copy()

# 2. Pegar a "Lista Telefônica" (a 'df_mestra_codigos')
#    (Vou selecionar só as colunas que quero adicionar)
df_nomes_e_ufs = df_mestra_codigos[['Cod_IBGE', 'Nome_Mestre', 'UF_Mestre']]

# 3. Remover as colunas "sujas" e "incompletas" da base final
#    (Elas vieram do arquivo de População e estão cheias de NaNs)
try:
    df_final = df_final.drop(columns=['Nome_Municipio_Mestre', 'Sigla_Mestre'])
except KeyError:
    print("Colunas de nome/sigla já foram removidas em um teste anterior.")

# 4. A JUNÇÃO FINAL (O "Polimento")
#    - 'df_final': tabela com 180k+ linhas.
#    - 'df_nomes_e_ufs': "lista telefônica" com ~5570 linhas.
#    - on='Cod_IBGE': A chave (6 dígitos, string)
#    - how='left': "Mantenha todas as linhas da 'df_final' e
#                  traga as informações de 'Nome_Mestre' e 'UF_Mestre'
#                  correspondentes."
df_base_polida = pd.merge(
    df_final,
    df_nomes_e_ufs,
    on='Cod_IBGE',
    how='left' 
)

# 5. Inspecionar o resultado
print("\n--- BASE DE DADOS FINAL E POLIDA ---")
print("Primeiras 5 linhas (agora com nomes preenchidos):")
print(df_base_polida.head())

print("\nÚltimas 5 linhas:")
print(df_base_polida.tail())

# 6. Teste Rápido: Verificar se ainda há NaNs nos nomes
nans_nos_nomes = df_base_polida['Nome_Mestre'].isna().sum()
print(f"\nNúmero de 'NaN's restantes na coluna de nome: {nans_nos_nomes}")


--- BASE DE DADOS FINAL E POLIDA ---
Primeiras 5 linhas (agora com nomes preenchidos):
  Cod_IBGE   Ano  CFEM  Populacao  PIB VAB Ind. VAB Serv.  Arrecadacao  \
0   110001  1985   NaN        NaN    0      NaN       NaN          NaN   
1   110001  1992   NaN   34768.00  NaN      NaN       NaN          NaN   
2   110001  1993   NaN   37036.00  NaN      NaN       NaN          NaN   
3   110001  1994   NaN   39325.00  NaN      NaN       NaN          NaN   
4   110001  1995   NaN   41574.00  NaN      NaN       NaN          NaN   

   Vinculos_totais  Empregos_mineracao  Empregos_servicos Mort_infant  \
0              NaN                 NaN                NaN         NaN   
1              NaN                 NaN                NaN         NaN   
2              NaN                 NaN                NaN         NaN   
3              NaN                 NaN                NaN         NaN   
4              NaN                 NaN                NaN         NaN   

             Nome_Mestre UF_

In [66]:
# --- 8: INSPECIONANDO OS 6 'NaN's ---

# 1. Filtra a base polida e mostra apenas as linhas onde o nome é 'NaN'
linhas_com_nan = df_base_polida[df_base_polida['Nome_Mestre'].isna()]

print("--- LINHAS COM NOMES FALTANTES (NaN) ---")
print(linhas_com_nan)

--- LINHAS COM NOMES FALTANTES (NaN) ---
       Cod_IBGE   Ano     CFEM  Populacao  PIB VAB Ind. VAB Serv.  \
2658     130009  2003 55503.03        NaN  NaN      NaN       NaN   
62997    290000  2003  3148.06        NaN  NaN      NaN       NaN   
148523   412792  2003  1103.26        NaN  NaN      NaN       NaN   
159453   430000  2003  2627.49        NaN  NaN      NaN       NaN   
159454   430000  2004  1096.18        NaN  NaN      NaN       NaN   
192846   530000  2003  4971.69        NaN  NaN      NaN       NaN   

        Arrecadacao  Vinculos_totais  Empregos_mineracao  Empregos_servicos  \
2658            NaN              NaN                 NaN                NaN   
62997           NaN              NaN                 NaN                NaN   
148523          NaN              NaN                 NaN                NaN   
159453          NaN              NaN                 NaN                NaN   
159454          NaN              NaN                 NaN                NaN   
1

In [67]:
# --- 9: VERIFICANDO 'NaN's NA COLUNA 'Sigla_Mestre' ---

# 1. Pegar a base final polida
df_para_teste = df_base_polida.copy()

# 2. Contar quantos 'NaN's (vazios) existem na coluna
nan_siglas = df_para_teste['UF_Mestre'].isna().sum()

# 3. Contar o total de linhas
total_linhas = len(df_para_teste)

# 4. Calcular a porcentagem preenchida
porcentagem_preenchida = ((total_linhas - nan_siglas) / total_linhas) * 100

# 5. Imprimir os resultados
print(f"--- Análise da Coluna 'UF_Mestre' ---")
print(f"Total de linhas na tabela: {total_linhas}")
print(f"Linhas com 'UF_Mestre' faltando (NaN): {nan_siglas}")
print(f"Porcentagem de preenchimento: {porcentagem_preenchida:.2f}%")

--- Análise da Coluna 'UF_Mestre' ---
Total de linhas na tabela: 192882
Linhas com 'UF_Mestre' faltando (NaN): 6
Porcentagem de preenchimento: 100.00%


In [68]:
# --- 10: RECARREGANDO O CHECKPOINT ---

import pandas as pd
from unidecode import unidecode # (Apenas para garantir que está na memória)

# 1. Definir o caminho para o arquivo que SALVAMOS
caminho_base_limpa = r'C:\Users\fabio\TCC\BASE_DE_DADOS_FINAL.csv'

print(f"Recarregando a base de dados limpa de: {caminho_base_limpa}")

# 2. Recarregar a base "limpa" (antes de qualquer cálculo de PIB per capita)
df_base_limpa = pd.read_csv(caminho_base_limpa)

print("Base de dados limpa recarregada.")

# --- INSPEÇÃO ---
print("\n--- Inspecionando valores 'sujos' na coluna PIB (da base limpa) ---")

# 3. Criar uma versão numérica da coluna
pib_numerico = pd.to_numeric(df_base_limpa['PIB'], errors='coerce')

# 4. Encontrar as linhas que falharam
linhas_que_falharam = df_base_limpa[pd.isna(pib_numerico)]
linhas_que_sucederam = df_base_limpa[~pd.isna(pib_numerico)]

# 5. Inspecionar os valores "sujos"
print(f"Total de linhas na tabela: {len(df_base_limpa)}")
print(f"Linhas que FALHARAM na conversão para número: {len(linhas_que_falharam)}")
print(f"Linhas que SUCEDERAM na conversão (ex: 2010): {len(linhas_que_sucederam)}")

# 6. Mostrar uma amostra dos valores que causaram o erro
if not linhas_que_falharam.empty:
    print("\n--- AMOSTRA DE VALORES 'SUJOS' (que viraram NaN) ---")
    # .unique() mostra todos os valores de texto únicos que falharam
    print(linhas_que_falharam['PIB'].unique()[:50]) # Mostra os 50 primeiros
else:
    print("\n--- Nenhuma linha falhou? ---")

# 7. Mostrar uma amostra dos valores que funcionaram
if not linhas_que_sucederam.empty:
    print("\n--- AMOSTRA DE VALORES 'LIMPOS' (que funcionaram) ---")
    print(linhas_que_sucederam['PIB'].unique()[:50]) # Mostra os 50 primeiros
else:
    print("\n--- Nenhuma linha funcionou? ---")

Recarregando a base de dados limpa de: C:\Users\fabio\TCC\BASE_DE_DADOS_FINAL.csv
Base de dados limpa recarregada.

--- Inspecionando valores 'sujos' na coluna PIB (da base limpa) ---
Total de linhas na tabela: 192882
Linhas que FALHARAM na conversão para número: 184963
Linhas que SUCEDERAM na conversão (ex: 2010): 7919

--- AMOSTRA DE VALORES 'SUJOS' (que viraram NaN) ---
[nan '109281,1781' '177564,2465' '179895,7262' '183537,0711' '211424,5989'
 '238481,6187' '268871,4545' '240400,546' '227408,5257' '242204,0093'
 '289674,1688' '278632,8003' '258967,5459' '281407,8291' '271545,8643'
 '278694,885' '288924,2054' '303373,2644' '297009,2016' '292204,6451'
 '278560,6611' '300924,0599' '342855,4829' '892725,6679' '481989,1154'
 '627530,0577' '700682,4918' '677973,2824' '854112,058' '898555,9164'
 '1015573,87' '1077412,506' '1065200,035' '1145687,427' '1238950,814'
 '1228539,426' '1525024,436' '1457069,732' '1431898,157' '1417476,33'
 '1397506,425' '1385714,386' '1400013,85' '1443340,329' '

In [69]:
# --- 11: A LIMPEZA CORRETA E O CÁLCULO ---

# 1. Recarregar a base limpa (o 'df_base_limpa' do Recomeço)

print("--- Aplicando a limpeza final (trocando ',' por '.') ---")

# 2. LIMPAR A COLUNA 'PIB'
#    - .astype(str) garante que estamos lidando com texto
df_base_limpa['PIB_limpo'] = df_base_limpa['PIB'].astype(str).str.replace(',', '.', regex=False)

# 3. LIMPAR A COLUNA 'Populacao'
#    (Mesma lógica, por segurança)
df_base_limpa['Pop_limpa'] = df_base_limpa['Populacao'].astype(str).str.replace(',', '.', regex=False)

# 4. AGORA, converter para NÚMERO
df_base_limpa['PIB'] = pd.to_numeric(df_base_limpa['PIB_limpo'], errors='coerce')
df_base_limpa['Populacao'] = pd.to_numeric(df_base_limpa['Pop_limpa'], errors='coerce')

# 5. Criar a nova coluna (AGORA VAI FUNCIONAR \o/)
df_base_limpa['PIB_per_capita'] = df_base_limpa['PIB'] / df_base_limpa['Populacao']

# 6. Inspecionar o resultado
print("\n--- Tabela com a NOVA COLUNA 'PIB_per_capita' ---")
codigo_araxa = '310400' # Código IBGE de 6 dígitos
df_araxa = df_base_limpa[df_base_limpa['Cod_IBGE'] == codigo_araxa]

print(f"\n--- Spot-Check em Araxá (Cód: {codigo_araxa}) ---")
# Mostrar as colunas relevantes
print(df_araxa[['Ano', 'PIB', 'Populacao', 'PIB_per_capita']])

--- Aplicando a limpeza final (trocando ',' por '.') ---

--- Tabela com a NOVA COLUNA 'PIB_per_capita' ---

--- Spot-Check em Araxá (Cód: 310400) ---
Empty DataFrame
Columns: [Ano, PIB, Populacao, PIB_per_capita]
Index: []


In [70]:
# --- 12: INSPECIONANDO A CHAVE 'Cod_IBGE' (DE NOVO) ---

print("--- Inspecionando 'Cod_IBGE' da base recarregada ---")

# 1. Mostrar o TIPO de dado da coluna
print(f"Tipo da coluna 'Cod_IBGE': {df_base_limpa['Cod_IBGE'].dtype}")

# 2. Mostrar uma amostra dos valores
print(f"Amostra dos valores: {df_base_limpa['Cod_IBGE'].unique()[:5]}")

--- Inspecionando 'Cod_IBGE' da base recarregada ---
Tipo da coluna 'Cod_IBGE': int64
Amostra dos valores: [110001 110002 110003 110004 110005]


In [71]:
# --- 13: A LIMPEZA CONDICIONAL ---

# 1. Copiar a base
df_base_calculo = df_base_limpa.copy()

# 2. Definir a função de "limpeza inteligente"
def limpar_numero_misto(valor):
    # Primeiro, transforma em texto
    valor_str = str(valor)
    
    # SE tiver vírgula, é formato BR (1.000,00)
    if ',' in valor_str:
        valor_limpo = valor_str.replace('.', '', regex=False).replace(',', '.', regex=False)
    # SENÃO, é formato EUA (1000.00) ou já está limpo
    else:
        valor_limpo = valor_str
        
    # Converte para número, e o que falhar vira NaN
    return pd.to_numeric(valor_limpo, errors='coerce')

print("--- Aplicando limpeza condicional no PIB e População ---")

# 3. Aplicar a função de limpeza (usando .apply())
df_base_calculo['PIB'] = df_base_calculo['PIB'].apply(limpar_numero_misto)
df_base_calculo['Populacao'] = df_base_calculo['Populacao'].apply(limpar_numero_misto)

# 4. Criar a nova coluna (AGORA VAI FUNCIONAR)
df_base_calculo['PIB_per_capita'] = df_base_calculo['PIB'] / df_base_calculo['Populacao']

print("Cálculo de PIB per capita concluído.")

--- Aplicando limpeza condicional no PIB e População ---
Cálculo de PIB per capita concluído.


In [72]:
# --- 14: "SPOT-CHECK" (TESTE DE AMOSTRA) PARA ARAXÁ ---

# 1. Definir o código que queremos "espionar"
codigo_araxa = 310400 

# 2. Filtrar a tabela
#    (O 'df_base_calculo' tem 'Cod_IBGE' como int64)
df_araxa = df_base_calculo[df_base_calculo['Cod_IBGE'] == codigo_araxa]

# 3. Inspecionar o resultado
print(f"--- MOSTRANDO DADOS PARA ARAXÁ (Cód: {codigo_araxa}) ---")
print(f"Encontradas {len(df_araxa)} linhas de dados para este município.")

# 4. Imprimir a tabela inteira para Araxá
#    (Agora com os dados de PIB e População corretos)
print(df_araxa[['Ano', 'PIB', 'Populacao', 'PIB_per_capita']])

--- MOSTRANDO DADOS PARA ARAXÁ (Cód: 310400) ---
Encontradas 35 linhas de dados para este município.
        Ano        PIB  Populacao  PIB_per_capita
78974  1985 1435479.57        NaN             NaN
78975  1992        NaN   71317.00             NaN
78976  1993        NaN   72695.00             NaN
78977  1994        NaN   73959.00             NaN
78978  1995        NaN   75193.00             NaN
78979  1996  892920.10        NaN             NaN
78980  1997        NaN   75072.00             NaN
78981  1998        NaN   75805.00             NaN
78982  1999 1070229.03   76536.00           13.98
78983  2000 1326569.85   77270.00           17.17
78984  2001 1373842.17   79945.00           17.18
78985  2002 1576730.36   80909.00           19.49
78986  2003 1655499.10   81796.00           20.24
78987  2004 1940181.69   83659.00           23.19
78988  2005 1833926.03   84689.00           21.65
78989  2006 1907427.97   85713.00           22.25
78990  2007 2613969.97        NaN             NaN

In [73]:
# --- 15: RECARGA, LIMPEZA, CÁLCULO E TESTE ---

# 1. RECARREGAR O CHECKPOINT (para garantir um recomeço limpo)
caminho_base_limpa = r'C:\Users\fabio\TCC\BASE_DE_DADOS_FINAL.csv'
print(f"Recarregando a base de dados limpa de: {caminho_base_limpa}")
df_base = pd.read_csv(caminho_base_limpa)
print("Base de dados limpa recarregada.")

# 2. Definir a função de "limpeza inteligente"
def limpar_numero_misto(valor):
    valor_str = str(valor)
    
    # SE tiver vírgula, é formato BR (1.000,00)
    if ',' in valor_str:
        # [A CORREÇÃO]: Removido o 'regex=False'
        valor_limpo = valor_str.replace('.', '').replace(',', '.')
    # SENÃO, é formato EUA (1000.00) ou já está limpo
    else:
        valor_limpo = valor_str
        
    return pd.to_numeric(valor_limpo, errors='coerce')

print("--- Aplicando limpeza condicional no PIB e População ---")

# 3. Aplicar a função de limpeza (usando .apply())
df_base['PIB'] = df_base['PIB'].apply(limpar_numero_misto)
df_base['Populacao'] = df_base['Populacao'].apply(limpar_numero_misto)

# 4. Criar a nova coluna (AGORA VAI FUNCIONAR)
df_base['PIB_per_capita'] = df_base['PIB'] / df_base['Populacao']
print("Cálculo de PIB per capita concluído.")

# 5. [FORMATAÇÃO] Configurar o Pandas para não usar notação científica
pd.set_option('display.float_format', '{:.2f}'.format)
print("Formatação de exibição definida.")

# 6. [O TESTE] Fazer o Spot-Check em Araxá
#    (Usando o NÚMERO 310400)
codigo_araxa = 310400 

#    (Nota: 'Cod_IBGE' foi lido como int64)
df_araxa = df_base[df_base['Cod_IBGE'] == codigo_araxa]

print(f"\n--- Spot-Check em Araxá (Cód: {codigo_araxa}) ---")
# Vamos mostrar as colunas relevantes
print(df_araxa[['Ano', 'PIB', 'Populacao', 'PIB_per_capita']])

Recarregando a base de dados limpa de: C:\Users\fabio\TCC\BASE_DE_DADOS_FINAL.csv
Base de dados limpa recarregada.
--- Aplicando limpeza condicional no PIB e População ---
Cálculo de PIB per capita concluído.
Formatação de exibição definida.

--- Spot-Check em Araxá (Cód: 310400) ---
        Ano        PIB  Populacao  PIB_per_capita
78974  1985 1435479.57        NaN             NaN
78975  1992        NaN   71317.00             NaN
78976  1993        NaN   72695.00             NaN
78977  1994        NaN   73959.00             NaN
78978  1995        NaN   75193.00             NaN
78979  1996  892920.10        NaN             NaN
78980  1997        NaN   75072.00             NaN
78981  1998        NaN   75805.00             NaN
78982  1999 1070229.03   76536.00           13.98
78983  2000 1326569.85   77270.00           17.17
78984  2001 1373842.17   79945.00           17.18
78985  2002 1576730.36   80909.00           19.49
78986  2003 1655499.10   81796.00           20.24
78987  2004 194

In [74]:
# --- 16: RECARREGAR A BASE E RENOMEAR O VAB ---

# Importar o Pandas (só para garantir que está na célula)
import pandas as pd

# Recarrego a base consolidada para garantir que as alterações 
# sejam aplicadas sobre o conjunto de dados final.
caminho_base = r'C:\Users\fabio\TCC\BASE_DE_DADOS_FINAL.csv'
df_base_final = pd.read_csv(caminho_base)

# Realizo o rename das colunas de produto para VAB (Valor Adicionado Bruto).
# Essa alteração é necessária para manter a precisão técnica da análise,
# diferenciando a produção setorial do PIB municipal.
df_base_final = df_base_final.rename(columns={
    'PIB': 'VAB_Total',
    'PIB_per_capita': 'VAB_per_capita'
})

# Verificação rápida da estrutura das colunas após o rename
print("Colunas atualizadas:", df_base_final.columns.tolist())

Recarregando a base de dados de: C:\Users\fabio\TCC\BASE_DE_DADOS_FINAL.csv

--- Colunas renomeadas com sucesso! ---
['Cod_IBGE', 'Ano', 'CFEM', 'Populacao', 'VAB_Total', 'VAB Ind.', 'VAB Serv.', 'Arrecadacao', 'Vinculos_totais', 'Empregos_mineracao', 'Empregos_servicos', 'Mort_infant', 'Nome_Mestre', 'UF_Mestre']


In [75]:
# --- 17: CARREGANDO E JUNTANDO O PIB REAL ---

# Defino o caminho e carrego os dados de PIB Real já processados anteriormente.
caminho_pib_real = r'C:\Users\fabio\TCC\FINALIZADOS\12_PIB_FINAL.csv'
df_pib_real = pd.read_csv(caminho_pib_real)

# Padronização de chaves: Para garantir o sucesso do merge, converto as colunas 
# de junção ('Cod_IBGE' e 'Ano') para string em ambos os DataFrames. 
# Isso evita erros de inconsistência entre tipos numéricos e objetos.
df_base_final['Cod_IBGE'] = df_base_final['Cod_IBGE'].astype(str)
df_base_final['Ano'] = df_base_final['Ano'].astype(str)

df_pib_real['Cod_IBGE'] = df_pib_real['Cod_IBGE'].astype(str)
df_pib_real['Ano'] = df_pib_real['Ano'].astype(str)

# Realizo a junção final utilizando o método 'outer' para preservar a 
# integridade de todas as observações de ambas as bases, garantindo que 
# nenhum dado municipal ou temporal seja perdido no processo.
df_base_final = pd.merge(
    df_base_final,
    df_pib_real,
    on=['Cod_IBGE', 'Ano'],
    how='outer'
)

# Verificação da estrutura final da base consolidada
print("Junção concluída. Resumo da base:")
print(df_base_final.info())

Carregando o arquivo de PIB Real de: C:\Users\fabio\TCC\FINALIZADOS\12_PIB_FINAL.csv
Padronizando chaves (Cod_IBGE e Ano) para string...

--- JUNÇÃO FINAL COM O PIB REAL CONCLUÍDA ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192882 entries, 0 to 192881
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Cod_IBGE             192882 non-null  object 
 1   Ano                  192882 non-null  object 
 2   CFEM                 52529 non-null   float64
 3   Populacao            153262 non-null  float64
 4   VAB_Total            139311 non-null  object 
 5   VAB Ind.             132874 non-null  object 
 6   VAB Serv.            132874 non-null  object 
 7   Arrecadacao          110839 non-null  float64
 8   Vinculos_totais      109960 non-null  float64
 9   Empregos_mineracao   109960 non-null  float64
 10  Empregos_servicos    109960 non-null  float64
 11  Mort_infant          161559 non-null 

In [76]:
# --- 18: VALIDAÇÃO E SALVAMENTO ---

# Configuro o Pandas para exibir valores decimais com duas casas, 
# evitando a notação científica e facilitando a leitura de grandes montantes.
pd.set_option('display.float_format', '{:.2f}'.format)

# Realizo um "Spot-Check" nos dados de Araxá (Cód: 310400).
# Esta validação é essencial para garantir que a integração do PIB Real 
# e o rename das variáveis ocorreram corretamente na unidade de análise principal.
codigo_araxa = '310400' 
df_araxa = df_base_final[df_base_final['Cod_IBGE'] == codigo_araxa]

colunas_validacao = ['Ano', 'VAB_Total', 'PIB_Real_Mil_Reais', 'PIB_per_capita_Real']
print(f"\n--- Validação: Dados de Araxá ---")
print(df_araxa[colunas_validacao])

# Exportação da Base de Dados Mestra:
# Salvo o dataset consolidado em dois formatos estratégicos:
# 1. .csv para portabilidade e arquivamento.
# 2. .dta para processamento imediato no Stata, onde realizarei as estimações econométricas.
caminho_csv = r'C:\Users\fabio\TCC\BASE_DE_DADOS_MESTRA_FINAL.csv'
caminho_stata = r'C:\Users\fabio\TCC\BASE_DE_DADOS_MESTRA_FINAL.dta'

df_base_final.to_csv(caminho_csv, index=False)
df_base_final.to_stata(caminho_stata, write_index=False)

print(f"\nBase Mestra consolidada e exportada com sucesso.")

Formatação de exibição definida.

--- Spot-Check em Araxá (Cód: 310400) ---
        Ano    VAB_Total  PIB_Real_Mil_Reais  PIB_per_capita_Real
78974  1985  1435479,572                 NaN                  NaN
78975  1992          NaN                 NaN                  NaN
78976  1993          NaN                 NaN                  NaN
78977  1994          NaN                 NaN                  NaN
78978  1995          NaN                 NaN                  NaN
78979  1996  892920,0956                 NaN                  NaN
78980  1997          NaN                 NaN                  NaN
78981  1998          NaN                 NaN                  NaN
78982  1999  1070229,031                 NaN                  NaN
78983  2000  1326569,853                 NaN                  NaN
78984  2001  1373842,173                 NaN                  NaN
78985  2002  1576730,362           829969.13             10170.32
78986  2003  1655499,098           994225.57             12031.82


C:\Users\fabio\AppData\Local\Temp\ipykernel_11292\1684722484.py:28: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    VAB Ind.   ->   VAB_Ind_
    VAB Serv.   ->   VAB_Serv_

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  df_base_final.to_stata(


Base de dados Mestra (STATA) FINAL salva com sucesso em: C:\Users\fabio\TCC\BASE_DE_DADOS_MESTRA_FINAL.dta


In [77]:
# --- 19: TESTE FINAL DE ARAXÁ (TODOS OS DADOS) ---

# Realizo uma inspeção completa em todos os indicadores da base consolidada
# especificamente para o município de Araxá (Cód: 310400). 
# Esta etapa de auditoria é fundamental para garantir a consistência 
# das séries temporais antes do início das estimações por Controle Sintético.

codigo_araxa = '310400' 
df_araxa = df_base_final[df_base_final['Cod_IBGE'] == codigo_araxa]

# Exibição integral dos dados de Araxá para conferência de nulos e outliers
print("--- Auditoria Final: Série Temporal de Araxá ---")
print(df_araxa)

--- Spot-Check Final em Araxá (Cód: 310400) ---
      Cod_IBGE   Ano        CFEM  Populacao    VAB_Total          VAB Ind.  \
78974   310400  1985         NaN        NaN  1435479,572               NaN   
78975   310400  1992         NaN   71317.00          NaN               NaN   
78976   310400  1993         NaN   72695.00          NaN               NaN   
78977   310400  1994         NaN   73959.00          NaN               NaN   
78978   310400  1995         NaN   75193.00          NaN               NaN   
78979   310400  1996         NaN        NaN  892920,0956  586740,580442328   
78980   310400  1997         NaN   75072.00          NaN               NaN   
78981   310400  1998         NaN   75805.00          NaN               NaN   
78982   310400  1999         NaN   76536.00  1070229,031  513339,509115522   
78983   310400  2000         NaN   77270.00  1326569,853  792927,646144769   
78984   310400  2001         NaN   79945.00  1373842,173  769485,287675057   
78985   310400  

In [78]:
# --- 20: A LIMPEZA FINAL (VABs e População) ---

def limpar_formato_monetario(valor):
    """
    Função para padronizar strings numéricas. 
    Converte o padrão brasileiro (1.000,00) para o padrão float (1000.00),
    garantindo a integridade da conversão para tipos numéricos.
    """
    valor_str = str(valor)
    if ',' in valor_str:
        # Remove separador de milhar e ajusta o decimal
        valor_limpo = valor_str.replace('.', '').replace(',', '.')
    else:
        valor_limpo = valor_str
    return pd.to_numeric(valor_limpo, errors='coerce')

# Aplico a limpeza nas colunas identificadas com inconsistências de tipo (strings/objetos).
# Este processo é vital para variáveis de valor adicionado, população e saúde.
colunas_para_limpar = ['VAB_Total', 'VAB Ind.', 'VAB Serv.', 'Populacao', 'Mort_infant']

for coluna in colunas_para_limpar:
    df_base_final[coluna] = df_base_final[coluna].apply(limpar_formato_monetario)

# Com os dados tipados corretamente, realizo o cálculo do VAB per capita.
# Utilizo esta métrica como um indicador de produtividade e bem-estar econômico municipal.
df_base_final['VAB_per_capita'] = df_base_final['VAB_Total'] / df_base_final['Populacao']

print("Padronização numérica e cálculo de indicadores concluídos."

--- Aplicando limpeza final nas colunas 'sujas' ---
Limpando coluna: VAB_Total
Limpando coluna: VAB Ind.
Limpando coluna: VAB Serv.
Limpando coluna: Populacao
Limpando coluna: Mort_infant
Limpeza concluída.
Cálculo de VAB_per_capita concluído.


In [79]:
# --- 21: O TESTE FINAL DE ARAXÁ ---

# Ajusto a formatação de exibição para duas casas decimais, facilitando 
# a conferência visual de grandes magnitudes financeiras e demográficas.
pd.set_option('display.float_format', '{:.2f}'.format)

# Realizo um "Spot-Check" específico para Araxá (Cód: 310400).
# O objetivo aqui é validar se a limpeza dos formatos numéricos e o cálculo 
# do VAB per capita estão consistentes com as séries de PIB Real.
codigo_araxa = '310400' 
df_araxa = df_base_final[df_base_final['Cod_IBGE'] == codigo_araxa]

# Seleciono as colunas críticas para a análise econométrica:
# Variáveis nominais limpas, população e os indicadores reais deflacionados.
colunas_validacao = [
    'Ano', 'VAB_Total', 'Populacao', 'VAB_per_capita', 
    'PIB_Real_Mil_Reais', 'PIB_per_capita_Real'
]

print(f"--- Validação de Indicadores: Unidade de Tratamento (Araxá) ---")
print(df_araxa[colunas_validacao])

Formatação de exibição definida.

--- Spot-Check em Araxá (Cód: 310400) ---
        Ano  VAB_Total  Populacao  VAB_per_capita  PIB_Real_Mil_Reais  \
78974  1985 1435479.57        NaN             NaN                 NaN   
78975  1992        NaN   71317.00             NaN                 NaN   
78976  1993        NaN   72695.00             NaN                 NaN   
78977  1994        NaN   73959.00             NaN                 NaN   
78978  1995        NaN   75193.00             NaN                 NaN   
78979  1996  892920.10        NaN             NaN                 NaN   
78980  1997        NaN   75072.00             NaN                 NaN   
78981  1998        NaN   75805.00             NaN                 NaN   
78982  1999 1070229.03   76536.00           13.98                 NaN   
78983  2000 1326569.85   77270.00           17.17                 NaN   
78984  2001 1373842.17   79945.00           17.18                 NaN   
78985  2002 1576730.36   80909.00           19.4

In [80]:
# --- 22: INSPEÇÃO INTEGRAÇ DE ARAXÁ (TODAS AS COLUNAS) ---

Realizo uma auditoria completa em todas as variáveis disponíveis para o 
# município de Araxá (Cód: 310400). Esta visualização integral é o passo 
# final de validação para garantir que não existam NaNs ou outliers 
# inesperados em nenhuma das dimensões da base consolidada.

codigo_araxa = 310400 

# Filtro o dataset completo para a unidade de tratamento principal
df_araxa_completo = df_base_final[df_base_final['Cod_IBGE'] == codigo_araxa]

# Exibição da série histórica completa com todas as colunas tratadas
print(f"--- Relatório Consolidado: Araxá (Série Histórica) ---")
print(df_araxa_completo)

--- MOSTRANDO TODOS OS DADOS PARA ARAXÁ (Cód: 310400) ---
Empty DataFrame
Columns: [Cod_IBGE, Ano, CFEM, Populacao, VAB_Total, VAB Ind., VAB Serv., Arrecadacao, Vinculos_totais, Empregos_mineracao, Empregos_servicos, Mort_infant, Nome_Mestre, UF_Mestre, PIB_Real_Mil_Reais, PIB_per_capita_Real, VAB_per_capita]
Index: []


In [81]:
# --- 23: VERIFICANDO FORMATOS ---

import pandas as pd

# Carrego a base consolidada para uma verificação final de estrutura.
# Note que forço a leitura de 'Cod_IBGE' e 'Ano' como strings (objetos) 
# para preservar zeros à esquerda e garantir a compatibilidade com 
# chaves primárias em modelos econométricos.
caminho_final = r'C:\Users\fabio\TCC\BASE_DE_DADOS_MESTRA_FINAL.csv'
df_teste = pd.read_csv(caminho_final, dtype={'Cod_IBGE': str, 'Ano': str})

# Inspeção técnica dos tipos de dados (dtypes):
# O objetivo aqui é garantir que variáveis monetárias e demográficas 
# foram lidas como floats ou ints, permitindo cálculos matemáticos imediatos.
print("\n--- Diagnóstico de Estrutura de Dados (Info) ---")
df_teste.info()

Carregando base de: C:\Users\fabio\TCC\BASE_DE_DADOS_MESTRA_FINAL.csv

--- INFORMAÇÕES DAS COLUNAS (DTYPES) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192882 entries, 0 to 192881
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Cod_IBGE             192882 non-null  object 
 1   Ano                  192882 non-null  object 
 2   CFEM                 52529 non-null   float64
 3   Populacao            153262 non-null  float64
 4   VAB_Total            139311 non-null  object 
 5   VAB Ind.             132874 non-null  object 
 6   VAB Serv.            132874 non-null  object 
 7   Arrecadacao          110839 non-null  float64
 8   Vinculos_totais      109960 non-null  float64
 9   Empregos_mineracao   109960 non-null  float64
 10  Empregos_servicos    109960 non-null  float64
 11  Mort_infant          161559 non-null  object 
 12  Nome_Mestre          192876 non-null  object 
 13  UF_Mest

In [82]:
# --- 24: ALTERAR FORMATOS DE LETRA PRA NÚMERO ---
import pandas as pd
import numpy as np

# Carrego a base consolidada garantindo a integridade das chaves primárias.
# Manter 'Cod_IBGE' e 'Ano' como string evita a perda de zeros à esquerda.
caminho_final = r'C:\Users\fabio\TCC\BASE_DE_DADOS_MESTRA_FINAL.csv'
df_analise = pd.read_csv(caminho_final, dtype={'Cod_IBGE': str, 'Ano': str})

def limpar_padrao_br(valor):
    """
    Realiza o parsing de strings no padrão brasileiro (1.000,00) para 
    o padrão float (1000.00). Caso o valor já seja numérico ou nulo, 
    a função mantém a integridade do dado original.
    """
    if isinstance(valor, str):
        # Normalização: remove separadores de milhar e ajusta o separador decimal
        valor_limpo = valor.replace('.', '').replace(',', '.')
        return pd.to_numeric(valor_limpo, errors='coerce')
    return valor

# Aplico a normalização nas variáveis econômicas e de saúde.
# Este tratamento é pré-requisito para o cálculo de indicadores e modelos regressivos.
cols_alvo = ['VAB_Total', 'VAB Ind.', 'VAB Serv.', 'Mort_infant']

for col in cols_alvo:
    if col in df_analise.columns:
        df_analise[col] = df_analise[col].apply(limpar_padrao_br)

# Auditoria técnica: Validando se as colunas foram convertidas para float64
print("\n--- Diagnóstico de Tipagem Pós-Conversão ---")
print(df_analise[cols_alvo].info())

# Validação visual da série histórica da unidade de tratamento (Araxá)
pd.set_option('display.float_format', '{:.2f}'.format)
df_araxa = df_analise[df_analise['Cod_IBGE'] == '310400']

print("\n--- Verificação de Consistência: Araxá (Últimos Registros) ---")
print(df_araxa[['Ano', 'VAB_Total', 'Mort_infant']].tail())

Carregando base de dados...
Convertendo colunas de texto para número...

--- Verificação de Tipos (Esperamos float64 para as colunas acima) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192882 entries, 0 to 192881
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   VAB_Total    139311 non-null  float64
 1   VAB Ind.     132874 non-null  float64
 2   VAB Serv.    132874 non-null  float64
 3   Mort_infant  124680 non-null  float64
dtypes: float64(4)
memory usage: 5.9 MB
None

--- Verificação de Valores em Araxá (310400) ---
        Ano  VAB_Total  Mort_infant
79004  2021 4178328.09        21.00
79005  2022        NaN        12.00
79006  2023        NaN         8.00
79007  2024        NaN        16.00
79008  2025        NaN          NaN


In [83]:
# --- 24.1: RECUPERANDO OS DADOS ---

import pandas as pd
import numpy as np

# 1. CARREGAMENTO DO CHECKPOINT
# Recupero a base mestra garantindo a tipagem das chaves primárias (id_municipio e id_ano).
path_input = r'C:\Users\fabio\TCC\BASE_DE_DADOS_MESTRA_FINAL.csv'
df_final = pd.read_csv(path_input, dtype={'Cod_IBGE': str, 'Ano': str})

def normalizar_numericos(valor):
    """
    Normaliza strings no padrão decimal ABNT (vírgula) para float (ponto).
    Essencial para a conversão de dados brutos de fontes públicas brasileiras.
    """
    valor_str = str(valor)
    if ',' in valor_str:
        valor_limpo = valor_str.replace('.', '').replace(',', '.')
    else:
        valor_limpo = valor_str
    return pd.to_numeric(valor_limpo, errors='coerce')

# 2. LIMPEZA DE VARIÁVEIS CRÍTICAS
# Aplico a normalização em todas as colunas que servirão de base para os cálculos de indicadores.
cols_to_clean = [
    'VAB Total', 'VAB Ind.', 'VAB Serv.', 'CFEM', 'Arrecadacao', 
    'Populacao', 'Vinculos_totais', 'Empregos_mineracao', 'PIB_per_capita_Real'
]

# Padronizo nomes de colunas caso existam variações (snake_case vs espaço)
df_final.columns = [c.replace(' ', '_') for c in df_final.columns]
cols_to_clean = [c.replace(' ', '_') for c in cols_to_clean]

for col in cols_to_clean:
    if col in df_final.columns:
        df_final[col] = df_final[col].apply(normalizar_numericos)

# 3. FEATURE ENGINEERING: CÁLCULO DE INDICADORES ECONÔMICOS
# Calculo as métricas de dependência fiscal, participação setorial e crescimento.
df_final['Dep_CFEM_Pct'] = (df_final['CFEM'] / df_final['Arrecadacao']) * 100
df_final['Part_Emp_Mineracao_Pct'] = (df_final['Empregos_mineracao'] / df_final['Vinculos_totais']) * 100

if 'VAB_Ind.' in df_final.columns and 'VAB_Total' in df_final.columns:
    df_final['Part_VAB_Ind_Pct'] = (df_final['VAB_Ind.'] / df_final['VAB_Total']) * 100

# Ordenação temporal para cálculo de taxas de variação (crescimento anual)
df_final = df_final.sort_values(by=['Cod_IBGE', 'Ano'])
df_final['Cresc_PIB_pc_Real'] = df_final.groupby('Cod_IBGE')['PIB_per_capita_Real'].pct_change() * 100
df_final['Cresc_Emprego_Total'] = df_final.groupby('Cod_IBGE')['Vinculos_totais'].pct_change() * 100

# 4. PADRONIZAÇÃO DE NOMENCLATURA (DICIONÁRIO STATA)
# Renomeio as variáveis para um padrão 'snake_case' curto, facilitando a manipulação no Stata.
map_stata = {
    'Cod_IBGE': 'id_municipio_6', 'Ano': 'id_ano', 
    'Nome_Mestre': 'id_nome_municipio', 'UF_Mestre': 'id_uf',
    'Populacao': 'bruto_populacao', 'VAB_Total': 'bruto_vab_total', 
    'VAB_Ind.': 'bruto_vab_industria', 'VAB_Serv.': 'bruto_vab_servicos',
    'CFEM': 'bruto_cfem_valor', 'Arrecadacao': 'bruto_arrecadacao_total',
    'Vinculos_totais': 'bruto_emp_formal_total', 'Empregos_mineracao': 'bruto_emp_mineracao',
    'PIB_per_capita_Real': 'ind_pib_pc_real_mil', 'Part_VAB_Ind_Pct': 'ind_part_vab_industria',
    'Cresc_PIB_pc_Real': 'ind_cresc_pib_pc_real_mil', 'Dep_CFEM_Pct': 'ind_dependencia_cfem'
}

df_organizada = df_final.rename(columns=map_stata)
df_organizada = df_organizada.replace([np.inf, -np.inf], np.nan) # Tratamento de infinitos (divisão por zero)

# 5. AUDITORIA FINAL E EXPORTAÇÃO
pd.set_option('display.float_format', '{:.2f}'.format)
print("\n--- Validação Final: Série Histórica de Araxá (Indicadores) ---")
print(df_organizada[df_organizada['id_municipio_6'] == '310400'].tail(10))

path_stata = r'C:\Users\fabio\TCC\BASE_STATA_FINAL.dta'
df_organizada.to_stata(path_stata, write_index=False)
print(f"\nPipeline concluído. Base exportada para: {path_stata}")

Carregando base de: C:\Users\fabio\TCC\BASE_DE_DADOS_MESTRA_FINAL.csv
Aplicando limpeza nos números (VABs, etc)...
Limpeza concluída. Recalculando variáveis...
Variáveis calculadas.
Renomeando colunas...


C:\Users\fabio\AppData\Local\Temp\ipykernel_11292\38292003.py:64: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_final['Cresc_PIB_pc_Real'] = df_final.groupby('Cod_IBGE')['PIB_per_capita_Real'].pct_change() * 100
C:\Users\fabio\AppData\Local\Temp\ipykernel_11292\38292003.py:65: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_final['Cresc_Emprego_Total'] = df_final.groupby('Cod_IBGE')['Vinculos_totais'].pct_change() * 100



--- VERIFICAÇÃO FINAL (ARAXÁ) ---
      id_ano  bruto_vab_total  bruto_vab_industria  ind_part_vab_industria
78994   2011       2665380.84            928856.24                   34.85
78995   2012       3368787.72           1384324.09                   41.09
78996   2013       3387936.92           1366714.42                   40.34
78997   2014       3508979.28           1383684.67                   39.43
78998   2015       3361644.44           1260186.85                   37.49
78999   2016       3022699.44            966544.45                   31.98
79000   2017       3147155.45           1019468.02                   32.39
79001   2018       3622619.65           1377818.86                   38.03
79002   2019       3621041.37           1460538.31                   40.33
79003   2020       3214746.25           1245242.02                   38.74
79004   2021       4178328.09           1994392.13                   47.73
79005   2022              NaN                  NaN               

In [84]:
# --- 25: VERIFICAÇÃO FINAL DE FORMATOS (DTYPES) ---

# Realizo a auditoria final da estrutura do DataFrame após o processo de renaming e 
# feature engineering. Este passo é crucial para garantir que todas as variáveis 
# categóricas (IDs) e métricas contínuas (floats) estejam corretamente tipadas 
# antes da exportação definitiva.

print("--- Diagnóstico de Estrutura: Base Final Consolidada ---")

# 1. Inspeção detalhada de memória e nulos
# O comando .info() valida a contagem de registros não-nulos e o uso de memória,
# permitindo identificar se há gaps de dados em colunas específicas de Araxá ou controles.
df_organizada.info()

print("\n" + "="*50 + "\n")

# 2. Sumário de Atributos (Dtypes)
# Verificação rápida para assegurar que id_municipio_6 e id_ano permanecem 
# como 'object' (strings) e os indicadores como 'float64'.
print("--- Sumário Técnico de Tipos de Dados ---")
print(df_organizada.dtypes)

--- Tipos de Dados da Base Final Organizada ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192882 entries, 0 to 192881
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   id_municipio_6             192882 non-null  object 
 1   id_ano                     192882 non-null  object 
 2   id_nome_municipio          192876 non-null  object 
 3   id_uf                      192876 non-null  object 
 4   bruto_populacao            153262 non-null  float64
 5   bruto_vab_total            139311 non-null  float64
 6   bruto_vab_industria        132874 non-null  float64
 7   bruto_vab_servicos         132874 non-null  float64
 8   bruto_cfem_valor           52529 non-null   float64
 9   bruto_arrecadacao_total    110839 non-null  float64
 10  bruto_emp_formal_total     109960 non-null  float64
 11  bruto_emp_mineracao        109960 non-null  float64
 12  bruto_emp_servicos         109960 non-

In [85]:
# --- RECORTE TEMPORAL E EXPORTAÇÃO DEFINITIVA (ARQUIVO DE ANÁLISE) ---

# Nesta etapa final, realizo o filtro do horizonte temporal da amostra. 
# Removo anos que apresentam inconsistências de coleta ou que estão fora do 
# escopo da análise do Controle Sintético (ex: 1985 e projeções de 2025).

print(f"Volume de registros (Pré-filtro): {len(df_organizada)}")

# 1. DEFINIÇÃO DO INTERVALO DA AMOSTRA
# Utilizo o operador '~' junto ao '.isin()' para excluir períodos que 
# poderiam enviesar a estimação do contrafactual devido a quebras estruturais.
anos_excluidos = ['1985', '1992', '1993', '1994', '1995', '2025']
df_organizada = df_organizada[~df_organizada['id_ano'].isin(anos_excluidos)]

print("-" * 30)
print(f"Volume de registros (Pós-filtro): {len(df_organizada)}")
print(f"Horizonte temporal consolidado: {sorted(df_organizada['id_ano'].unique())}")

# 2. GERAÇÃO DOS ARQUIVOS FINAIS PARA ESTIMAÇÃO
# Sobrescrevo os arquivos com a base 'limpa' e pronta para processamento.
# O arquivo .dta é otimizado para o Stata, preservando a tipagem definida anteriormente.
path_csv = r'C:\Users\fabio\TCC\BASE_STATA_FINAL.csv'
path_dta = r'C:\Users\fabio\TCC\BASE_STATA_FINAL.dta'

df_organizada.to_csv(path_csv, index=False)
df_organizada.to_stata(path_dta, write_index=False)

print("\n--- PIPELINE DE TRATAMENTO EM PYTHON CONCLUÍDO ---")
print("Status: Base de dados pronta para análise econométrica no Stata.")

Total de linhas ANTES da remoção: 192882
Anos presentes antes: ['1985', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
------------------------------
Total de linhas DEPOIS da remoção: 170494
Anos presentes depois: ['1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

Salvando versão final (sem 1985/2025) em: C:\Users\fabio\TCC\BASE_STATA_FINAL.dta
--- PROJETO PYTHON CONCLUÍDO! ---
Pode abrir o Stata agora.


In [86]:
# --- INVENTÁRIO FINAL E DICIONÁRIO DE VARIÁVEIS (DATA DICTIONARY) ---

# Este bloco gera a listagem definitiva de todas as variáveis tratadas e 
# calculadas durante o pipeline. Este inventário serve como referência 
# técnica para a correta especificação do modelo econométrico no Stata.

print(f"--- INVENTÁRIO DE VARIÁVEIS CONSOLIDADAS (Total: {len(df_organizada.columns)}) ---")
print("Variáveis prontas para análise no Stata:\n")

# Listagem estruturada das colunas para conferência de nomenclatura (Snake Case)
for coluna in df_organizada.columns:
    print(f"  > [V] {coluna}")

print("\n" + "="*50)
print("Pipeline de Dados: Status FINALIZADO.")
print("A base de dados foi exportada seguindo os padrões de tipagem e recorte temporal.")

--- INVENTÁRIO DA BASE DE DADOS (21 Variáveis) ---
Copie a lista abaixo para suas anotações:

- id_municipio_6
- id_ano
- id_nome_municipio
- id_uf
- bruto_populacao
- bruto_vab_total
- bruto_vab_industria
- bruto_vab_servicos
- bruto_cfem_valor
- bruto_arrecadacao_total
- bruto_emp_formal_total
- bruto_emp_mineracao
- bruto_emp_servicos
- bruto_pib_real
- ind_mortalidade_infantil
- ind_pib_pc_real_mil
- ind_part_vab_industria
- ind_part_emp_mineracao
- ind_cresc_pib_pc_real_mil
- ind_cresc_emp_total
- ind_dependencia_cfem

Verificação concluída. Tudo pronto pro Stata!


In [87]:
# --- GERAÇÃO DE AGREGADOS REGIONAIS (BENCHMARKS: MG E BRASIL) ---

import pandas as pd
import numpy as np

# Carrego a base final consolidada para gerar as médias de comparação.
caminho_stata = r'C:\Users\fabio\TCC\BASE_STATA_FINAL.csv'
df_completa = pd.read_csv(caminho_stata)

# Automatizo a identificação de variáveis de estoque (brutas) para a agregação.
colunas_brutas = [c for c in df_completa.columns if c.startswith('bruto_')]

def gerar_benchmark_regional(df_input, nome_regiao, sigla_uf):
    """
    Agrega dados municipais para criar séries temporais de nível estadual ou nacional.
    Recalcula indicadores de intensidade (per capita) e participação para evitar 
    o erro de média das médias, garantindo precisão estatística.
    """
    # A. Agregação Temporal: Soma dos valores brutos e média dos indicadores sociais
    df_agg = df_input.groupby('id_ano').agg({
        **{col: 'sum' for col in colunas_brutas},
        'ind_mortalidade_infantil': 'mean'
    }).reset_index()
    
    # B. Recálculo de Indicadores de Produtividade e Bem-estar
    # Calculo o PIB e VAB per capita real utilizando as somas agregadas.
    df_agg['ind_pib_pc_real_mil'] = df_agg['bruto_pib_real'] / df_agg['bruto_populacao']
    df_agg['ind_vab_pc'] = df_agg['bruto_vab_total'] / df_agg['bruto_populacao']
    
    # Participações Setoriais e Dependência Fiscal (CFEM)
    df_agg['ind_part_vab_industria'] = (df_agg['bruto_vab_industria'] / df_agg['bruto_vab_total']) * 100
    df_agg['ind_part_emp_mineracao'] = (df_agg['bruto_emp_mineracao'] / df_agg['bruto_emp_formal_total']) * 100
    df_agg['ind_dependencia_cfem'] = (df_agg['bruto_cfem_valor'] / df_agg['bruto_arrecadacao_total']) * 100
    
    # C. Cálculo de Taxas de Crescimento Anual
    df_agg = df_agg.sort_values('id_ano')
    df_agg['ind_cresc_pib_pc_real_mil'] = df_agg['ind_pib_pc_real_mil'].pct_change() * 100
    df_agg['ind_cresc_emp_total'] = df_agg['bruto_emp_formal_total'].pct_change() * 100
    
    # D. Metadados de Identificação
    df_agg['id_municipio_6'] = '000000' # Código fictício para agregados
    df_agg['id_nome_municipio'] = nome_regiao
    df_agg['id_uf'] = sigla_uf
    
    return df_agg.replace([np.inf, -np.inf], np.nan)

# --- EXECUÇÃO DO PROCESSAMENTO ---

# Geração do benchmark para o Estado de Minas Gerais (Filtro por prefixo IBGE '31')
mask_mg = df_completa['id_municipio_6'].astype(str).str.startswith('31', na=False)
df_minas = gerar_benchmark_regional(df_completa[mask_mg], 'Minas Gerais', 'MG')

# Geração do benchmark nacional (Brasil)
df_brasil = gerar_benchmark_regional(df_completa, 'Brasil', 'BR')

# --- VALIDAÇÃO TÉCNICA (SPOT-CHECK) ---
print(f"PIB per Capita Real (BR): {df_brasil.iloc[-1]['ind_pib_pc_real_mil']:.2f}")
print(f"PIB per Capita Real (MG): {df_minas.iloc[-1]['ind_pib_pc_real_mil']:.2f}")

# --- EXPORTAÇÃO DOS ARQUIVOS DE COMPARAÇÃO ---
caminho_mg = r'C:\Users\fabio\TCC\DADOS_AGREGADOS_MG.dta'
caminho_br = r'C:\Users\fabio\TCC\DADOS_AGREGADOS_BR.dta'

df_minas.to_stata(caminho_mg, write_index=False)
df_brasil.to_stata(caminho_br, write_index=False)

print("\nBenchmarks regionais exportados com sucesso para análise comparativa.")

Carregando base final...

--- Gerando MG ---
--- Gerando BRASIL ---

--- VERIFICAÇÃO: PIB PER CAPITA (Último Ano) ---
BRASIL: nan
MINAS:  nan

Arquivos salvos com nomes atualizados!


C:\Users\fabio\AppData\Local\Temp\ipykernel_11292\4073873614.py:38: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_agg['ind_cresc_pib_pc_real_mil'] = df_agg['ind_pib_pc_real_mil'].pct_change() * 100
C:\Users\fabio\AppData\Local\Temp\ipykernel_11292\4073873614.py:38: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_agg['ind_cresc_pib_pc_real_mil'] = df_agg['ind_pib_pc_real_mil'].pct_change() * 100
